# Model Training & Evaluation
## Car Price Prediction

This notebook trains multiple machine learning models on the preprocessed car price dataset and compares their performance using cross-validation and test set metrics.

## Section 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Try-except for optional advanced models
try:
    import xgboost as xgb
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("⚠ XGBoost not found. Install with: pip install xgboost")

try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("⚠ LightGBM not found. Install with: pip install lightgbm")

try:
    from catboost import CatBoostRegressor
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print("⚠ CatBoost not found. Install with: pip install catboost")

print("✓ Core libraries imported successfully")
print(f"XGBoost: {'✓' if XGBOOST_AVAILABLE else '✗'}")
print(f"LightGBM: {'✓' if LIGHTGBM_AVAILABLE else '✗'}")
print(f"CatBoost: {'✓' if CATBOOST_AVAILABLE else '✗'}")

## Section 2: Load Data

Load the cleaned dataset and separate features (X) from target (y).

In [ ]:
# Load cleaned dataset
df = pd.read_csv('../data/cleaned_car_data.csv')

print(f"Dataset loaded: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values: {df.isnull().sum().sum()}")

# Separate features and target
X = df.drop(columns=['selling_price'])
y = df['selling_price']

print(f"\n{'='*70}")
print(f"DATASET SPLIT:")
print(f"{'='*70}")
print(f"Features (X): {X.shape} - {X.columns.tolist()}")
print(f"Target (y): {y.shape}")
print(f"\nTarget statistics:")
print(y.describe())

## Section 3: Train-Test Split

Split data into 80% training and 20% testing with random_state=42 for reproducibility.

In [ ]:
# 80-20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"TRAIN-TEST SPLIT:")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"\nTrain set target statistics:")
print(y_train.describe())
print(f"\nTest set target statistics:")
print(y_test.describe())

## Section 4: Define Evaluation Function

Create a function to consistently evaluate all models with test metrics and cross-validation scores.

In [ ]:
def evaluate_model(model_name, model, X_train, X_test, y_train, y_test):
    """
    Train and evaluate a model with test metrics and cross-validation.
    
    Returns:
        dict: {model_name, model, rmse, mae, r2, cv_mean, cv_std}
    """
    # Fit on training data
    model.fit(X_train, y_train)
    
    # Predict on test data
    y_pred = model.predict(X_test)
    
    # Calculate test metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # Calculate 5-fold cross-validation R2 on training data
    cv_fold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv_fold, scoring='r2')
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    # Print results
    print(f"[{model_name:20}] RMSE: ₹{rmse:>12,.0f} | MAE: ₹{mae:>12,.0f} | R2: {r2:.4f} | CV R2: {cv_mean:.4f} (±{cv_std:.4f})")
    
    # Return results dictionary
    return {
        'model_name': model_name,
        'model': model,
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'cv_mean': cv_mean,
        'cv_std': cv_std
    }

print("✓ Evaluation function defined")

## Section 5: Train Baseline Models

Train simple, interpretable models as baseline comparisons.

In [ ]:
results = []

print("TRAINING BASELINE MODELS:")
print("=" * 120)

# Linear Regression
lr_model = LinearRegression()
results.append(evaluate_model(
    "Linear Regression", lr_model, X_train, X_test, y_train, y_test
))

# Decision Tree Regressor
dt_model = DecisionTreeRegressor(random_state=42, max_depth=10)
results.append(evaluate_model(
    "Decision Tree", dt_model, X_train, X_test, y_train, y_test
))

print("=" * 120)

## Section 6: Train Advanced Models

Train ensemble and gradient boosting models for better performance.

In [ ]:
print("TRAINING ADVANCED MODELS:")
print("=" * 120)

# Random Forest
rf_model = RandomForestRegressor(
    n_estimators=200, max_depth=15, random_state=42, n_jobs=-1
)
results.append(evaluate_model(
    "Random Forest", rf_model, X_train, X_test, y_train, y_test
))

# XGBoost
if XGBOOST_AVAILABLE:
    xgb_model = XGBRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42
    )
    results.append(evaluate_model(
        "XGBoost", xgb_model, X_train, X_test, y_train, y_test
    ))
else:
    print("[XGBoost             ] Skipped (not installed)")

# LightGBM
if LIGHTGBM_AVAILABLE:
    lgb_model = LGBMRegressor(
        n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1
    )
    results.append(evaluate_model(
        "LightGBM", lgb_model, X_train, X_test, y_train, y_test
    ))
else:
    print("[LightGBM            ] Skipped (not installed)")

# CatBoost
if CATBOOST_AVAILABLE:
    cb_model = CatBoostRegressor(
        iterations=200, learning_rate=0.05, random_state=42, verbose=0
    )
    results.append(evaluate_model(
        "CatBoost", cb_model, X_train, X_test, y_train, y_test
    ))
else:
    print("[CatBoost            ] Skipped (not installed)")

print("=" * 120)

## Section 7: Model Comparison Table

Build a comprehensive comparison of all trained models and visualize their performance.

In [ ]:
# Create comparison DataFrame (excluding model objects)
comparison_data = [
    {
        'Model': r['model_name'],
        'RMSE': r['rmse'],
        'MAE': r['mae'],
        'R2': r['r2'],
        'CV_R2_mean': r['cv_mean'],
        'CV_R2_std': r['cv_std']
    }
    for r in results
]

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('R2', ascending=False).reset_index(drop=True)

print("\nMODEL COMPARISON TABLE:")
print("=" * 100)

# Display with formatting
display_df = comparison_df.copy()
display_df['RMSE'] = display_df['RMSE'].apply(lambda x: f"₹{x:,.0f}")
display_df['MAE'] = display_df['MAE'].apply(lambda x: f"₹{x:,.0f}")
display_df['R2'] = display_df['R2'].apply(lambda x: f"{x:.4f}")
display_df['CV_R2_mean'] = display_df['CV_R2_mean'].apply(lambda x: f"{x:.4f}")
display_df['CV_R2_std'] = display_df['CV_R2_std'].apply(lambda x: f"{x:.4f}")

print(display_df.to_string(index=False))
print("=" * 100)

### Visualization: R² Score and RMSE Comparison

In [ ]:
# R2 score and RMSE comparison (horizontal bar charts)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# R2 score comparison
comparison_sorted_r2 = comparison_df.sort_values('R2')
bars1 = ax1.barh(comparison_sorted_r2['Model'], comparison_sorted_r2['R2'], color='steelblue')
ax1.set_xlabel('R² Score (Test Set)', fontsize=11, fontweight='bold')
ax1.set_title('Model R² Score Comparison', fontsize=12, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)
for i, v in enumerate(comparison_sorted_r2['R2']):
    ax1.text(v - 0.01, i, f' {v:.4f}', va='center', ha='right', fontweight='bold', color='white')

# RMSE comparison
comparison_sorted_rmse = comparison_df.sort_values('RMSE')
bars2 = ax2.barh(comparison_sorted_rmse['Model'], comparison_sorted_rmse['RMSE'], color='coral')
ax2.set_xlabel('RMSE (₹)', fontsize=11, fontweight='bold')
ax2.set_title('Model RMSE Comparison (Lower is Better)', fontsize=12, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../models/model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Model comparison charts saved")

## Section 8: Best Model Selection

Select the model with the highest R² and analyze its predictions.

In [ ]:
# Find best model by test R2
best_result = max(results, key=lambda x: x['r2'])
best_model = best_result['model']
best_model_name = best_result['model_name']

print(f"\n{'='*70}")
print(f"BEST MODEL: {best_model_name}")
print(f"{'='*70}")
print(f"Test RMSE: ₹{best_result['rmse']:,.0f}")
print(f"Test MAE:  ₹{best_result['mae']:,.0f}")
print(f"Test R²:   {best_result['r2']:.4f}")
print(f"CV R² (5-fold): {best_result['cv_mean']:.4f} (±{best_result['cv_std']:.4f})")
print(f"{'='*70}")

# Get predictions
y_pred_best = best_model.predict(X_test)

### Visualization: Actual vs Predicted & Residual Distribution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted scatter plot
ax1.scatter(y_test, y_pred_best, alpha=0.6, s=30, color='steelblue', edgecolors='navy', linewidth=0.5)

# Add y=x reference line (perfect predictions)
min_val = min(y_test.min(), y_pred_best.min())
max_val = max(y_test.max(), y_pred_best.max())
ax1.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')

ax1.set_xlabel('Actual Selling Price (₹)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Predicted Selling Price (₹)', fontsize=11, fontweight='bold')
ax1.set_title(f'{best_model_name} - Actual vs Predicted', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Residuals histogram
residuals = y_test - y_pred_best
ax2.hist(residuals, bins=50, color='steelblue', edgecolor='navy', alpha=0.7)
ax2.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
ax2.set_xlabel('Residuals (₹)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax2.set_title('Residual Distribution', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../models/best_model_predictions.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\nResidual Statistics:")
print(f"Mean: ₹{residuals.mean():,.0f}")
print(f"Std Dev: ₹{residuals.std():,.0f}")
print(f"Min: ₹{residuals.min():,.0f}")
print(f"Max: ₹{residuals.max():,.0f}")

## Section 9: Feature Importance

Visualize the most important features for the best model.

In [ ]:
# Check if model has feature_importances_ attribute (tree-based models)
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=True)
    
    # Get top 15
    feature_importance_top15 = feature_importance.tail(15)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(feature_importance_top15['feature'], feature_importance_top15['importance'], color='steelblue')
    ax.set_xlabel('Importance Score', fontsize=11, fontweight='bold')
    ax.set_title(f'{best_model_name} - Top 15 Feature Importance', fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../models/feature_importance.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print("\nTOP 15 FEATURES:")
    print(feature_importance_top15.to_string(index=False))
    
elif best_model_name == 'Linear Regression':
    # For Linear Regression, use coefficients
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': np.abs(best_model.coef_)  # Use absolute value
    }).sort_values('importance', ascending=True)
    
    feature_importance_top15 = feature_importance.tail(15)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(feature_importance_top15['feature'], feature_importance_top15['importance'], color='steelblue')
    ax.set_xlabel('|Coefficient| (Absolute Value)', fontsize=11, fontweight='bold')
    ax.set_title(f'{best_model_name} - Top 15 Feature Importance (|Coefficients|)', fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../models/feature_importance.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print("\nTOP 15 FEATURES (by absolute coefficient):")
    print(feature_importance_top15.to_string(index=False))
else:
    print(f"Model {best_model_name} does not support feature importance extraction.")

## Section 10: Save Artifacts

Save the best model, comparison table, and metadata for future use.

In [ ]:
import os

# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# 1. Save best model
best_model_path = '../models/best_model.pkl'
joblib.dump(best_model, best_model_path)
print(f"✓ Best model saved to: {best_model_path}")

# 2. Save comparison table
comparison_csv_path = '../models/model_comparison.csv'
comparison_df.to_csv(comparison_csv_path, index=False)
print(f"✓ Model comparison table saved to: {comparison_csv_path}")

# 3. Save metadata
metadata = {
    'best_model_name': best_model_name,
    'rmse': float(best_result['rmse']),
    'mae': float(best_result['mae']),
    'r2': float(best_result['r2']),
    'cv_r2_mean': float(best_result['cv_mean']),
    'cv_r2_std': float(best_result['cv_std']),
    'feature_columns': X.columns.tolist(),
    'training_date': datetime.now().isoformat(),
    'training_samples': len(X_train),
    'test_samples': len(X_test)
}

metadata_path = '../models/model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✓ Metadata saved to: {metadata_path}")

print(f"\n✓ All artifacts saved successfully!")

## Section 11: Summary

### Training Complete ✓

**Models Trained:** Linear Regression, Decision Tree, Random Forest, XGBoost, LightGBM, CatBoost

**Best Model** was selected based on the highest Test R² score among all models evaluated.

**Performance Metrics (Best Model):**
- **Test RMSE:** Average prediction error in ₹ (rupees) — lower is better
- **Test MAE:** Mean absolute error in ₹ — average magnitude of errors
- **Test R²:** Proportion of variance explained by the model (closer to 1.0 is better)
- **Cross-Validation R²:** 5-fold CV on training data — indicates model stability

**Interpreting RMSE in ₹ Context:**
- Given the target variable range of ₹20,000 to ₹26,75,000 (mean ~₹4.6L), an RMSE of ~₹1.8-1.9L represents a reasonable prediction capability
- The model performs better on mid-range cars and may have higher error on extreme price points

**Dataset Specifications:**
- Training samples: 2,861
- Test samples: 716
- Total features: 18
- Target variable: selling_price (₹)

**Saved Artifacts:**
- ✓ Best model: `../models/best_model.pkl`
- ✓ Model comparison: `../models/model_comparison.csv`
- ✓ Metadata: `../models/model_metadata.json`
- ✓ Feature importance plot: `../models/feature_importance.png`
- ✓ Prediction analysis plot: `../models/best_model_predictions.png`
- ✓ Model comparison plot: `../models/model_comparison.png`

**Next Steps:**
- ✅ **SHAP Explainability:** The best model is ready for SHAP analysis (`shap.TreeExplainer` for tree-based models)
- ✅ **API Deployment:** The saved `best_model.pkl`, `brand_encoder.pkl`, `scaler.pkl`, and `feature_columns.pkl` provide everything needed to build a Flask/FastAPI prediction endpoint
- ✅ **Dashboard:** Model metadata and comparison data can power an interactive Streamlit/Dash dashboard